# Econometric Analysis: Modal Shift and Carbon Taxation
This notebook mirrors the complete econometric pipeline for analyzing commuter travel mode choices under Carbon Surcharge (Loss) vs Green Mobility Voucher (Gain) framing under identical net financial advantages.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 1. Load Processed Experimental Datasets

In [ ]:
df_long = pd.read_csv('../data/processed/analysis_long.csv')
df_part = pd.read_csv('../data/processed/analysis_participant.csv')
print(f'Choice-level panel observations: {len(df_long)}')
print(f'Participant-level observations:   {len(df_part)}')
df_long.head()

## 2. Descriptive Choice Shares & Balance Verification

In [ ]:
# Mode shares by treatment branch
choice_summary = df_long.groupby('treatment')['choice_car'].agg(['count', 'mean', 'std']).reset_index()
choice_summary['pct_car'] = choice_summary['mean'] * 100
choice_summary['se'] = choice_summary['std'] / np.sqrt(choice_summary['count']) * 100
choice_summary

In [ ]:
# Bar plot of Private Car Share by Policy Framing
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(choice_summary['treatment'], choice_summary['pct_car'], yerr=choice_summary['se']*1.96, capsize=5, color=['#1e40af', '#dc2626', '#059669'], alpha=0.85)
ax.set_title('Private Car Choice Share by Policy Treatment (95% CI)', fontweight='bold')
ax.set_ylabel('Private Car Share (%)')
ax.set_ylim(0, 100)
ax.set_xticklabels(['A: Control', 'B: Loss (+€30 Tax)', 'C: Gain (-€30 Bonus)'])
plt.show()

## 3. Main Econometric Estimations (LPM & Logit with Clustered SEs)

In [ ]:
# Linear Probability Model (LPM) clustered by participant
model_lpm = smf.ols('choice_car ~ treat_B + treat_C + distance_km + cost_diff + time_diff', data=df_long).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_long['participant_code']}
)
print(model_lpm.summary().tables[1])

## 4. Key Hypothesis Test: Wald Test of Framing Equivalence (B = C)

In [ ]:
# Formal Wald Test of H0: beta_B = beta_C
wald_test = model_lpm.wald_test('treat_B = treat_C')
print('Wald Test H0: beta_B == beta_C')
print(f'F-statistic: {float(wald_test.statistic):.4f}')
print(f'p-value:     {float(wald_test.pvalue):.4f}')

## 5. Main Logit Model & Average Marginal Effects (AME)

In [ ]:
model_logit = smf.logit('choice_car ~ treat_B + treat_C + distance_km + cost_diff + time_diff', data=df_long).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_long['participant_code']}
)
print(model_logit.summary().tables[1])
margeff = model_logit.get_margeff()
print(margeff.summary())

## 6. Heterogeneity & Interaction Analysis

In [ ]:
# Treatment interactions with Car Ownership and NEP Ecological worldview
model_hetero = smf.ols(
    'choice_car ~ treat_B + treat_C + treat_B:owns_car + treat_C:owns_car + treat_B:nep_centered + treat_C:nep_centered + distance_km + cost_diff + owns_car + nep_score',
    data=df_long
).fit(cov_type='cluster', cov_kwds={'groups': df_long['participant_code']})
print(model_hetero.summary().tables[1])